In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
import zipfile, re, shutil
from glob import glob
import gc
from tqdm.auto import tqdm

## Starting off with a quick example to look at each monthly dataset's structure and testing a merge on a small subset

In [2]:
# final scratch directories - ERA5
pres_ex = Path("/home/scratch/ocarlisle/era5_483/pressure_levels/era5_pl_202001_09z_real.nc")
sl_ex = Path("/home/scratch/jtoniolo/483/single_levels/era5_sl_instant_202001_09z.nc")
accum_ex = Path("/home/scratch/jtoniolo/483/single_levels/era5_sl_accum_202001_09z.nc")

## Dropping the version number and whatever blank list 'number' is... throws error when trying to merge later

In [3]:
ds_pres = xr.open_dataset(pres_ex, drop_variables=['expver','number'])
ds_sl = xr.open_dataset(sl_ex, drop_variables=['expver','number'])
ds_accum = xr.open_dataset(accum_ex, drop_variables=['expver','number'])

### Notice how z is in geopotential. We want to change that to geopotential height, will do below.

In [4]:
ds_pres

<xarray.Dataset>
Dimensions:         (valid_time: 6, pressure_level: 2, latitude: 133,
                     longitude: 241)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 2020-01-02T09:00:00 ... 2020-...
  * pressure_level  (pressure_level) float64 850.0 500.0
  * latitude        (latitude) float64 53.0 52.75 52.5 52.25 ... 20.5 20.25 20.0
  * longitude       (longitude) float64 -125.0 -124.8 -124.5 ... -65.25 -65.0
Data variables:
    z               (valid_time, pressure_level, latitude, longitude) float32 ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-27T22:21 GRIB to CDM+CF via cfgrib-0.9.1...

### TCWV = total column integrated water vapor... which is essentially precipitable water (PWAT). We will change this below along with the units to the more familiar millimeters

In [5]:
ds_sl

<xarray.Dataset>
Dimensions:     (valid_time: 6, latitude: 133, longitude: 241)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2020-01-02T09:00:00 ... 2020-01-1...
  * latitude    (latitude) float64 53.0 52.75 52.5 52.25 ... 20.5 20.25 20.0
  * longitude   (longitude) float64 -125.0 -124.8 -124.5 ... -65.5 -65.25 -65.0
Data variables:
    tcwv        (valid_time, latitude, longitude) float32 ...
    cape        (valid_time, latitude, longitude) float32 ...
    msl         (valid_time, latitude, longitude) float32 ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-27T22:22 GRIB to CDM+CF via cfgrib-0.9.1...

### tp = total precipitation. Units are in meters so we will convert to millimeters.

In [6]:
ds_accum

<xarray.Dataset>
Dimensions:     (valid_time: 6, latitude: 133, longitude: 241)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2020-01-02T09:00:00 ... 2020-01-1...
  * latitude    (latitude) float64 53.0 52.75 52.5 52.25 ... 20.5 20.25 20.0
  * longitude   (longitude) float64 -125.0 -124.8 -124.5 ... -65.5 -65.25 -65.0
Data variables:
    tp          (valid_time, latitude, longitude) float32 ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-27T22:22 GRIB to CDM+CF via cfgrib-0.9.1...

## Merging example datasets

In [7]:
merged_ex = xr.merge([ds_pres,ds_sl,ds_accum])

#### Making the heights at different pressure levels its own variable

In [8]:
merged_ex['z850'] = merged_ex.z.sel(pressure_level=850)
merged_ex['z500'] = merged_ex.z.sel(pressure_level=500)

In [9]:
merged_ex = merged_ex.drop(['z'])

In [10]:
merged_ex

<xarray.Dataset>
Dimensions:         (valid_time: 6, pressure_level: 2, latitude: 133,
                     longitude: 241)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 2020-01-02T09:00:00 ... 2020-...
  * pressure_level  (pressure_level) float64 850.0 500.0
  * latitude        (latitude) float64 53.0 52.75 52.5 52.25 ... 20.5 20.25 20.0
  * longitude       (longitude) float64 -125.0 -124.8 -124.5 ... -65.25 -65.0
Data variables:
    tcwv            (valid_time, latitude, longitude) float32 ...
    cape            (valid_time, latitude, longitude) float32 ...
    msl             (valid_time, latitude, longitude) float32 ...
    tp              (valid_time, latitude, longitude) float32 ...
    z850            (valid_time, latitude, longitude) float32 ...
    z500            (valid_time, latitude, longitude) float32 ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-27T22:21 GRIB to CDM+CF via cfgrib-0.9.1...

In [13]:
from dask.diagnostics import ProgressBar

print("GB: ", merged_ex.nbytes / 1e9)

delayed = merged_ex.to_netcdf(
    "/home/z2044621/merge_test.nc",
    engine="netcdf4",
    compute=False
)

with ProgressBar():
    delayed.compute()
    
    
##### DELETE THIS LATER ######

GB:  0.004618688
[########################################] | 100% Completed | 109.28 ms


## Repeating the process for all of the ERA5 timesteps

In [14]:
sfc_files = sorted(glob("/home/scratch/jtoniolo/483/single_levels/era5_sl_instant_*_09z.nc"))
pres_files = sorted(glob("/home/scratch/ocarlisle/era5_483/pressure_levels/*_real.nc"))
accum_files = sorted(glob("/home/scratch/jtoniolo/483/single_levels/era5_sl_accum*.nc"))

In [15]:
ds_sfc = xr.open_mfdataset(sfc_files,combine='by_coords',drop_variables=['expver','number'])
ds_pres = xr.open_mfdataset(pres_files,combine='by_coords',drop_variables=['expver','number'])
ds_accum = xr.open_mfdataset(accum_files,combine='by_coords',drop_variables=['expver','number'])

#### Geopotential to geopotential height conversion

In [42]:
g0 = 9.80665
ds_pres['z'] = ds_pres['z'] / g0
ds_pres['z'].attrs['units'] = 'm' 

#### Changing to pwat along with its units

In [45]:
ds_sfc = ds_sfc.rename({'tcwv':'pwat'})

In [47]:
ds_sfc['pwat'].attrs['units'] = 'mm' 

#### Converting precip to millimeters and renaming

In [61]:
ds_accum['tp'] = ds_accum['tp'] / 1000
ds_accum['tp'].attrs['units'] = 'mm'

#### Final merge of all 3 ERA5 sources

In [63]:
merged = xr.merge([ds_sfc,ds_pres,ds_accum])
merged['z850'] = merged.z.sel(pressure_level=850)
merged['z500'] = merged.z.sel(pressure_level=500)
merged = merged.drop(['z'])

In [64]:
merged

<xarray.Dataset>
Dimensions:         (valid_time: 999, latitude: 133, longitude: 241,
                     pressure_level: 2)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 2020-01-02T09:00:00 ... 2025-...
  * latitude        (latitude) float64 53.0 52.75 52.5 52.25 ... 20.5 20.25 20.0
  * longitude       (longitude) float64 -125.0 -124.8 -124.5 ... -65.25 -65.0
  * pressure_level  (pressure_level) float64 850.0 500.0
Data variables:
    tcwv            (valid_time, latitude, longitude) float32 dask.array<chunksize=(6, 133, 241), meta=np.ndarray>
    cape            (valid_time, latitude, longitude) float32 dask.array<chunksize=(6, 133, 241), meta=np.ndarray>
    msl             (valid_time, latitude, longitude) float32 dask.array<chunksize=(6, 133, 241), meta=np.ndarray>
    tp              (valid_time, latitude, longitude) float32 dask.array<chunksize=(6, 133, 241), meta=np.ndarray>
    z850            (valid_time, latitude, longitude) float32 dask.array<chunksize=(6, 133, 241), meta=np.ndarray>
    z500            (valid_time, latitude, longitude) float32 dask.array<chunksize=(6, 133, 241), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-27T22:22 GRIB to CDM+CF via cfgrib-0.9.1...

In [ ]:
merged.to_netcdf('/home/scratch/jtoniolo/483/ERA5_vars.nc', engine='netcdf4')
print("Done")

In [18]:
print(ds_pres.valid_time.equals(ds_accum.valid_time))
print(ds_pres.latitude.equals(ds_accum.latitude))
print(ds_pres.longitude.equals(ds_accum.longitude))

True
True
True
